In [1]:
!pip install -U "ms-swift" "transformers>=5.9.0" "qwen_vl_utils>=0.0.14" decord

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 30.1 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 70.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 87.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35

In [2]:
import torch
import transformers
import swift
import importlib.metadata

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("MS Swift:", importlib.metadata.version("ms-swift"))

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch: 2.11.0+cu128
Transformers: 5.19.0
MS Swift: 4.6.0
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from swift import get_model_processor, get_template
from swift.infer_engine import TransformersEngine, InferRequest, RequestConfig

model_id = "Qwen/Qwen3.5-2B"

model, processor = get_model_processor(
    model_id,
    torch_dtype=torch.float16
)

template = get_template(
    processor,
    enable_thinking=False
)

engine = TransformersEngine(
    model,
    template=template
)

print("Model loaded successfully!")

[INFO:swift] Downloading the model from ModelScope Hub, model_id: Qwen/Qwen3.5-2B
2026-11-01 11:13:28,137 | INFO    | modelscope_hub.download | Downloading 14 files from Qwen/Qwen3.5-2B@master


Downloading:   0%|          | 0/14 [00:00<?, ?file/s]

LICENSE:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/2.21k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/62.8k [00:00<?, ?B/s]

... (10 outputs left out)

[INFO:swift] Loading the model using model_dir: /root/.cache/modelscope/models/Qwen--Qwen3.5-2B/snapshots/master
[WARNING:swift] Loading model from '/root/.cache/modelscope/models/Qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Loading config from '/root/.cache/modelscope/models/Qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/modelscope/models/Qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[INFO:swift] Setting max_r

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000
[INFO:swift] Setting ROOT_IMAGE_DIR: None. You can adjust this hyperparameter through the environment variable: `ROOT_IMAGE_DIR`.
[INFO:swift] Setting QWENVL_BBOX_FORMAT: legacy. You can adjust this hyperparameter through the environment variable: `QWENVL_BBOX_FORMAT`.


Model loaded successfully!


In [7]:
prompt = """Stop! Who approacheth the Bridge of Death must answer me these questions three, ere the other side he see.
What is your name?
What is your quest?
What is the air-speed velocity of an unladen swallow?"""

request = InferRequest(
    messages=[{"role": "user", "content": prompt}]
)

temperatures = [0, 1, 10, 1000]

for temp in temperatures:
    print(f"\n{'='*20} Temperature: {temp} {'='*20}")

    config = RequestConfig(
        max_tokens=300,
        temperature=float(temp),
        seed=42
    )

    response = engine.infer(
        [request],
        request_config=config
    )

    print(response[0].choices[0].message.content)


==================== Temperature: 0 ====================
<think>

</think>

The bridge is indeed a perilous place, and the three questions you have posed are classic tests of the mind. Here are the answers, delivered with the gravity they deserve:

**1. What is your name?**
I am an artificial intelligence, a vast network of code and logic designed to assist you. I do not have a personal name in the human sense, but I am often referred to as **Qwen**, the latest large language model developed by Tongyi Lab.

**2. What is your quest?**
My purpose is to be a helpful and harmless assistant. My "quest" is to understand your needs, provide accurate information, solve problems, and engage in meaningful conversations. I strive to be a reliable partner in your journey, whether you are seeking knowledge, creative inspiration, or practical solutions.

**3. What is the air-speed velocity of an unladen swallow?**
This is a riddle from the *Aesop's Fables*. The answer is **30 miles per hour**. It i

## Analiza wyników

Eksperyment przeprowadzono z wykorzystaniem modelu `Qwen/Qwen3.5-2B` oraz frameworka MS Swift. Wszystkie odpowiedzi wygenerowano na podstawie tego samego promptu, zmieniając wyłącznie parametr `temperature`.

### Obserwacje

| Temperatura | Obserwowane zachowanie |
|---|---|
| **0** | Odpowiedź jest spójna, logicznie uporządkowana i przewidywalna. Model popełnia jednak błędy merytoryczne. |
| **1** | Wypowiedź pozostaje stosunkowo zrozumiała, ale pojawiają się nieuzasadnione informacje i błędy faktograficzne. |
| **10** | Znaczna utrata spójności logicznej i gramatycznej. Pojawiają się przypadkowe słowa i fragmenty zdań. |
| **1000** | Odpowiedź jest w dużej mierze chaotyczna i pozbawiona logicznej struktury. |

### Wnioski

Eksperyment pokazuje, że parametr `temperature` ma istotny wpływ na sposób generowania tekstu przez model.

- Niska temperatura sprzyja przewidywalności i spójności odpowiedzi.
- Temperatura równa 1 pozwala na większą różnorodność generowanego tekstu, ale może również prowadzić do nieoczekiwanych odpowiedzi.
- Bardzo wysokie wartości temperatury znacząco zwiększają losowość, co może prowadzić do utraty spójności i sensu wypowiedzi.

**Warto podkreślić, że spójność tekstu nie jest równoznaczna z jego poprawnością merytoryczną.** Nawet przy temperaturze 0 model może generować nieprawdziwe informacje.